## Generating RAG Answers

In [2]:
# Now we evaluate the full RAG pipeline. For each generated question, we run RAG and save the answer produced by the LLM. 
# Later, we'll compare this answer with the original FAQ answer.

# This is the A->Q->A' setup:

# A = original answer in the FAQ
# Q = generated question from this answer
# A' = answer produced by our RAG system
# If A' is close to A, the RAG system is doing a good job.

# This is still offline evaluation. We can compare A and A' because our questions came from FAQ records. 
# For each question, we know which original answer it came from.

# Loading the data
# Create a new notebook for RAG evaluation.

# Load the ground truth questions:

import pandas as pd

df_ground_truth = pd.read_csv("ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
# Load the FAQ documents and the search index:

from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [4]:
# Create a lookup table for the original FAQ documents:

doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc
    
# We'll use this lookup table to find the original answer for each ground truth question.

In [5]:
# Running RAG
# Import the usual things first:

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()



In [6]:
# We use RAGWithUsage from the evaluation utilities. It subclasses RAGBase from module 01, so it has the same rag method.

# It stores token usage after each LLM call. Then we can calculate the total cost later.

# It also uses the search boosts we selected in the search tuning lesson: question=1.0, answer=2.0, and section=0.1.

from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [8]:
# For each question, RAGBase searches the FAQ, builds a prompt with the retrieved context, and asks the LLM to answer. 
# We save the answer so the next lesson can judge it.

# Run RAG for one question:

rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
question, answer_llm

('Can I still join the course if I just found it now?',
 'Yes, you can still join the course. If you want a certificate, you need to submit your project while submissions are still being accepted.')

In [9]:
ground_truth[0]

{'question': 'Can I still join the course if I just found it now?',
 'document': '74eb249bbf'}

In [10]:
# Check the cost of this call:

assistant.total_cost()

0.000996

In [12]:
# Get the original answer from the document ID:

doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [13]:
# Now save both answers in one record:

rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Can I still join the course if I just found it now?',
 'answer_llm': 'Yes, you can still join the course. If you want a certificate, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [14]:
# Processing all questions
# Create a function that processes one ground truth record:

def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [15]:
# Test it on one record:

answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Can I still join the course if I just found it now?',
 'answer_llm': 'Yes, you can still join the course. If you want a certificate, though, you need to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [16]:
# Before running the full batch, reset the usage we collected while testing:

assistant.reset_usage()

In [17]:
# This calls the LLM once per ground truth question, so it can take some time. Let's process the questions in parallel and track progress.

# Import the parallel processing helper from the same utility file:

from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

# Run RAG for all ground truth questions:

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)
    
# generate_rag_answer returns one answer record for each question.

# Collect the answer records:

answers = []

for answer_record in results:
    answers.append(answer_record)

  0%|          | 0/565 [00:00<?, ?it/s]

In [18]:
answers[10]

{'question': 'How do I join the Office Hours or live workshop if the Zoom link isn’t shared with students?',
 'answer_llm': 'The Zoom link is only shared with instructors/presenters/TAs, not students.\n\nIf you’re a student, join the Office Hours or live workshop via:\n- **YouTube Live** on the DataTalksClub YouTube channel\n- **Slido** for questions, with the link pinned in the live chat\n- Check the **announcements channel on Telegram and Slack** for the video URL before the session starts\n\nDon’t post questions in the chat, since they may be missed if the room is busy.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksCl

In [20]:
# Calculate the total cost:

assistant.total_cost()

0.6118424999999996

In [19]:
# Save the answers:

df_answers = pd.DataFrame(answers)
df_answers.to_csv("rag-answers-new.csv", index=False)

# We generated this file for the course materials on May 29, 2026. The run used 395 ground truth questions.

# The total cost was $0.611, about 61 cent.

# If you don't want to generate the RAG answers yourself, download the file we prepared:

# PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main

# wget -O data/rag-answers-new.csv ${PREFIX}/04-evaluation/data/rag-answers-new.csv